# Deploy FAST

In this notebook you will clone the FAST repository, create a `strands-travel-agent` pattern,
and deploy the full stack with CDK.

The `FAST-stack` name is the default set by the FAST CDK app. Module 2 and Module 3 use
`workshop-*` stack names because Workshop Studio provisions them from `contentspec.yaml`;
Module 4 differs because you deploy FAST yourself from the notebook/IDE.

Steps:

1. Clone the FAST repository into `/workshop/fast-agent`
2. Copy `strands-single-agent` into `strands-travel-agent` and overwrite the agent code
3. Update the Dockerfile, requirements, and `config.yaml`
4. `cdk bootstrap` (once per account/region), then `cdk deploy`
5. Deploy the Amplify frontend and create a Cognito user

The deployment takes ~5–10 minutes.

> **Demo logging:** the agent code written below streams full model events back to the
> caller for workshop visibility. Production systems should log metadata only — never
> full model responses.


## Step 1: Clone the FAST Repository

Clone into `/workshop/fast-agent` (idempotent — skip if the directory already exists).


In [ ]:
import os, pathlib, subprocess

WORKSHOP_DIR = pathlib.Path("/workshop")
WORKSHOP_DIR.mkdir(parents=True, exist_ok=True)

FAST_DIR = WORKSHOP_DIR / "fast-agent"
FAST_REPO = "https://github.com/awslabs/fullstack-solution-template-for-agentcore.git"
FAST_REF = "v0.4.1"  # pinned release — known-good for this workshop (SHA 9b1374c3)

if FAST_DIR.exists() and (FAST_DIR / ".git").exists():
    print(f"FAST repo already present at {FAST_DIR} — skipping clone")
else:
    subprocess.run(
        ["git", "clone", FAST_REPO, str(FAST_DIR)],
        check=True,
    )
    print(f"Cloned FAST into {FAST_DIR}")

current_ref = subprocess.run(
    ["git", "-C", str(FAST_DIR), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()

target_ref = subprocess.run(
    ["git", "-C", str(FAST_DIR), "rev-parse", f"{FAST_REF}^{{commit}}"],
    capture_output=True, text=True,
).stdout.strip()

if target_ref and current_ref == target_ref:
    print(f"Already on {FAST_REF} ({current_ref[:8]})")
else:
    subprocess.run(["git", "-C", str(FAST_DIR), "fetch", "--tags", "origin"], check=True)
    subprocess.run(["git", "-C", str(FAST_DIR), "checkout", FAST_REF], check=True)
    print(f"Checked out pinned ref {FAST_REF}")

os.chdir(FAST_DIR)
print(f"Current directory: {os.getcwd()}")


## Step 2: Create the Travel Agent Pattern

Copy `patterns/strands-single-agent` into `patterns/strands-travel-agent`, rename the agent
script, and overwrite it with the travel-specific code.


In [ ]:
import pathlib, shutil

FAST_DIR = pathlib.Path("/workshop/fast-agent")
SRC = FAST_DIR / "patterns" / "strands-single-agent"
DST = FAST_DIR / "patterns" / "strands-travel-agent"

if not SRC.exists():
    raise RuntimeError(f"{SRC} missing — FAST layout unexpected")

if DST.exists():
    print(f"{DST} already exists — leaving as-is")
else:
    shutil.copytree(SRC, DST)
    print(f"Copied pattern into {DST}")

basic = DST / "basic_agent.py"
travel = DST / "travel_agent.py"
if basic.exists() and not travel.exists():
    basic.rename(travel)
    print(f"Renamed basic_agent.py → travel_agent.py")
else:
    print(f"travel_agent.py already present: {travel.exists()}")

for child in sorted(DST.rglob('*'))[:20]:
    rel = child.relative_to(DST)
    print(f"  {rel}")


## Step 3: Overwrite `travel_agent.py`

This is the only file you need to write — FAST inherits gateway auth, memory wiring, and the
code interpreter. The agent calls the LLM Gateway when SSM parameters exist and falls back to
direct Bedrock otherwise. Notebook 03 will populate those SSM parameters.


In [ ]:
TRAVEL_AGENT_PY = r'''"""Travel Agent — Strands agent with Gateway MCP tools, Memory, and Code Interpreter."""

import json
import logging
import os

from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import (
    AgentCoreMemorySessionManager,
)
from bedrock_agentcore.runtime import BedrockAgentCoreApp, RequestContext
from strands import Agent
from strands.models import BedrockModel
from tools.gateway import create_gateway_mcp_client
from utils.auth import extract_user_id_from_context
from utils.ssm import get_ssm_parameter
from tools.code_interpreter import StrandsCodeInterpreterTools

logger = logging.getLogger(__name__)
app = BedrockAgentCoreApp()

SYSTEM_PROMPT = """\
You are a helpful travel planning assistant. You have access to the following
tools through an AgentCore Gateway:

Flight tools:
- search_flights(origin, destination, date, max_results?) – search flights by route and date
- get_flight_details(flight_id) – get full details for a specific flight
- search_flights_by_budget(origin, destination, date, max_price) – search within a budget

Hotel tools:
- search_hotels(city, checkin_date?, checkout_date?, guests?, max_results?) – search hotels by city
- get_hotel_details(hotel_id) – get full details for a specific hotel
- search_hotels_by_budget(city, checkin_date, checkout_date, max_price_per_night) – search within a budget

You also have access to a Code Interpreter for calculations and data analysis.

When planning a trip:
1. Parse the user's request for origin, destination, dates, budget, and preferences.
2. Search for flights matching the route and dates.
3. Search for hotels in the destination city for the stay dates.
4. If a budget is provided, use the budget-filtered search tools.
5. Compare options and recommend the best combination of flight + hotel.
6. Present a clear, structured itinerary with prices and key details.

Use IATA airport codes (e.g. LAX, LHR, SFO, TYO) for flight searches.
Use city names (e.g. London, Tokyo, Paris) for hotel searches.
Dates should be in YYYY-MM-DD format.
"""


def _create_model():
    """Create the model — LiteLLM Gateway if configured, else direct Bedrock.

    Model ID matches the one Module 2 step-2 primes in a fresh Workshop Studio account
    (Claude Sonnet 4.6). The `global.` cross-region inference profile is used so the agent
    works in any deploy region without per-region geo-prefix derivation. Using a different
    Anthropic model here would hit a Bedrock Marketplace subscription gate the first time
    the agent calls it.
    """
    stack_name = os.environ.get("STACK_NAME", "")
    try:
        gateway_url = get_ssm_parameter(f"/{stack_name}/llm_gateway_url")
        gateway_key = get_ssm_parameter(f"/{stack_name}/llm_gateway_key")
        if gateway_url and gateway_key:
            from strands.models.litellm import LiteLLMModel
            logger.info("[MODEL] Using LiteLLM Gateway: %s", gateway_url)
            return LiteLLMModel(
                model_id="bedrock/global.anthropic.claude-sonnet-4-6",
                api_base=gateway_url,
                api_key=gateway_key,
            )
    except Exception as e:
        logger.info("[MODEL] LLM Gateway not configured (%s), using direct Bedrock", e)
    return BedrockModel(
        model_id="global.anthropic.claude-sonnet-4-6", temperature=0.1
    )


def _create_session_manager(user_id, session_id):
    memory_id = os.environ.get("MEMORY_ID")
    if not memory_id:
        raise ValueError("MEMORY_ID environment variable is required")
    config = AgentCoreMemoryConfig(
        memory_id=memory_id, session_id=session_id, actor_id=user_id
    )
    return AgentCoreMemorySessionManager(
        agentcore_memory_config=config,
        region_name=os.environ.get("AWS_DEFAULT_REGION", "us-west-2"),
    )


def create_travel_agent(user_id, session_id):
    model = _create_model()
    session_manager = _create_session_manager(user_id, session_id)
    region = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")
    code_tools = StrandsCodeInterpreterTools(region)
    gateway_client = create_gateway_mcp_client()
    return Agent(
        name="travel_agent",
        system_prompt=SYSTEM_PROMPT,
        tools=[gateway_client, code_tools.execute_python_securely],
        model=model,
        session_manager=session_manager,
        trace_attributes={"user.id": user_id, "session.id": session_id},
    )


@app.entrypoint
async def invocations(payload, context: RequestContext):
    user_query = payload.get("prompt")
    session_id = payload.get("runtimeSessionId")
    if not all([user_query, session_id]):
        yield {"status": "error", "error": "Missing required fields: prompt or runtimeSessionId"}
        return
    try:
        user_id = extract_user_id_from_context(context)
        agent = create_travel_agent(user_id, session_id)
        async for event in agent.stream_async(user_query):
            yield json.loads(json.dumps(dict(event), default=str))
    except Exception as e:
        logger.exception("Agent run failed")
        yield {"status": "error", "error": str(e)}

if __name__ == "__main__":
    app.run()
'''

(DST / "travel_agent.py").write_text(TRAVEL_AGENT_PY)

import ast
ast.parse(TRAVEL_AGENT_PY)
print("travel_agent.py written and parses cleanly")

## Step 4: Update Dockerfile and requirements

The Dockerfile from `strands-single-agent` references `basic_agent.py` — rewrite it to point
at `travel_agent.py`, and append `litellm` so the LLM Gateway client works inside the container.


In [ ]:
dockerfile = DST / "Dockerfile"
text = dockerfile.read_text()
new_text = text.replace("strands-single-agent", "strands-travel-agent").replace("basic_agent", "travel_agent")
if new_text != text:
    dockerfile.write_text(new_text)
    print(f"Rewrote {dockerfile}")
else:
    print(f"{dockerfile} already references travel_agent")

req = DST / "requirements.txt"
existing = set(req.read_text().splitlines())
# Pinned to exact tested versions: litellm matches the Module 2 LLM Gateway pin;
# strands-agents matches the version FAST's pattern requirements.txt already pins.
additions = ["litellm==1.83.0", "strands-agents[litellm]==1.32.0"]
with req.open("a") as f:
    for line in additions:
        if line not in existing:
            f.write(line + "\n")
            print(f"Added '{line}' to requirements.txt")
        else:
            print(f"'{line}' already in requirements.txt")


## Step 5: Write `infra-cdk/config.yaml`

Tell FAST's CDK app to build the travel-agent pattern.


In [ ]:
CONFIG_YAML = '''stack_name_base: FAST-stack
admin_user_email:

backend:
  pattern: strands-travel-agent
  deployment_type: docker
  network_mode: PUBLIC
'''

config_path = FAST_DIR / "infra-cdk" / "config.yaml"
config_path.write_text(CONFIG_YAML)
print(f"Wrote {config_path}:")
print(config_path.read_text())


# Widen the AgentCore Runtime role's Token Vault scope BEFORE the first `cdk deploy`.
# FAST scopes `secretsmanager:GetSecretValue` on the runtime role to
# `oauth2/${stack_name_base}-runtime-gateway-auth*`. Adding a second OAuth2 provider in
# notebook 04a/04b (e.g. `workshop-tools-gateway-auth`) produces a Token Vault secret
# OUTSIDE that scope — AgentCore Runtime then fails at invoke time with:
#   AccessDeniedException: not authorized to perform secretsmanager:GetSecretValue
# Widening here at role-creation time bakes the wide resource into the inline policy.
# Workshop Studio learner roles cannot modify IAM after the fact, so this MUST happen
# before the first `cdk deploy`.
#
# Idempotency note: FAST v0.4.1 already contains a *different* wide pattern elsewhere in
# backend-stack.ts (on the OAuth2 provider Lambda's role, unrelated to the runtime role).
# So "wide in text" is already True before we patch — we must check whether the NARROW
# runtime pattern is still present, not whether the wide pattern exists.
import re
backend_stack = FAST_DIR / "infra-cdk" / "lib" / "backend-stack.ts"
text = backend_stack.read_text()
narrow_re = re.compile(r"bedrock-agentcore-identity!default/oauth2/[^\"\x27`]*runtime-gateway-auth\*")
wide = "bedrock-agentcore-identity!default/oauth2/*"

narrow_hits = narrow_re.findall(text)
if not narrow_hits:
    print(f"Already widened: {backend_stack} (no runtime-gateway-auth narrow pattern found)")
else:
    new_text, n = narrow_re.subn(wide, text)
    backend_stack.write_text(new_text)
    print(f"Widened Token Vault scope in {backend_stack} ({n} narrow pattern(s) replaced)")
    # Verify the write took effect
    verify = backend_stack.read_text()
    if narrow_re.search(verify):
        raise RuntimeError(
            f"Widening verification failed: narrow runtime-gateway-auth pattern still in {backend_stack}."
        )
    print(f"Verified: {backend_stack} no longer contains the narrow runtime pattern")


## Step 6: Install CDK Dependencies and Bootstrap

`cdk bootstrap` is only required once per account/region — the command is idempotent.


In [ ]:
import subprocess, os

INFRA_DIR = FAST_DIR / "infra-cdk"
os.chdir(INFRA_DIR)

# `npm ci` installs exactly what package-lock.json (committed in FAST v0.4.1)
# specifies — reproducible installs, no floating versions.
print("=== npm ci ===")
r = subprocess.run(["npm", "ci"], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(
        f"npm ci failed (exit {r.returncode}).\n"
        f"stdout: {r.stdout[-1500:]}\nstderr: {r.stderr[-1500:]}"
    )
print(r.stdout[-400:])

print("\n=== cdk bootstrap ===")
r = subprocess.run(["npx", "cdk", "bootstrap"], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(
        f"cdk bootstrap failed (exit {r.returncode}).\n"
        f"stdout: {r.stdout[-1500:]}\nstderr: {r.stderr[-1500:]}"
    )
print(r.stdout[-400:])

## Step 7: Authenticate Docker with ECR

CDK needs a valid ECR login to push the agent container image.


In [ ]:
import subprocess, os, pathlib, json
import boto3

account_id = subprocess.run(
    ["aws", "sts", "get-caller-identity", "--query", "Account", "--output", "text"],
    capture_output=True, text=True, check=True,
).stdout.strip()
region = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or boto3.session.Session().region_name
)
if not region:
    raise RuntimeError(
        "No AWS region configured. Set AWS_REGION (or AWS_DEFAULT_REGION), "
        "or run `aws configure set region <region>`."
    )
registry = f"{account_id}.dkr.ecr.{region}.amazonaws.com"

docker_cfg = pathlib.Path.home() / ".docker" / "config.json"
docker_cfg.parent.mkdir(parents=True, exist_ok=True)
docker_cfg.write_text("{}")

pw = subprocess.run(
    ["aws", "ecr", "get-login-password", "--region", region],
    capture_output=True, text=True, check=True,
).stdout.strip()

subprocess.run(
    ["docker", "login", "--username", "AWS", "--password-stdin", registry],
    input=pw, text=True, check=True,
)
print(f"Logged in to {registry}")

## Step 8: Deploy the Backend Stack

`cdk deploy` builds the agent container, pushes it to ECR, and provisions the AgentCore
Runtime, Memory, Gateway, and Cognito User Pool. Takes ~5–10 minutes.


In [ ]:
import subprocess, os

INFRA_DIR = FAST_DIR / "infra-cdk"
os.chdir(INFRA_DIR)

r = subprocess.run(
    ["npx", "cdk", "deploy", "--require-approval", "never"],
    capture_output=True, text=True,
)
if r.returncode != 0:
    raise RuntimeError(
        f"cdk deploy failed (exit {r.returncode}).\n"
        f"stdout tail: {r.stdout[-2500:]}\nstderr tail: {r.stderr[-2500:]}"
    )
print(r.stdout[-600:])
print("cdk deploy complete")

## Step 9: Deploy the Frontend

Return to the FAST repo root and run the Amplify deploy script.


In [ ]:
import subprocess, os

os.chdir(FAST_DIR)

# FAST's deploy-frontend.py uses `npm install` (not `npm ci`), which floats the
# frontend deps off the caret ranges instead of the committed package-lock.json.
# Recent vite 7.x builds default to the rolldown bundler, where the template's
# object-form `manualChunks` fails ("manualChunks is not a function"). Run
# `npm ci` first so the lockfile-pinned vite (7.3.1) is installed deterministically;
# deploy-frontend.py's later `npm install` then finds the satisfied lockfile and
# does not re-float. This keeps the FAST v0.4.1 pin from leaking.
print("=== npm ci (frontend, lockfile-exact) ===")
_fe = FAST_DIR / "frontend"
_r = subprocess.run(["npm", "ci"], cwd=str(_fe), capture_output=True, text=True)
if _r.returncode != 0:
    # Not fatal on its own — deploy-frontend.py will still try npm install — but
    # surface it so a real lockfile problem is visible.
    print("WARN: npm ci returned", _r.returncode, _r.stderr[-500:])
else:
    print("npm ci OK (frontend deps pinned to lockfile)")

r = subprocess.run(
    ["python3", "scripts/deploy-frontend.py"],
    capture_output=True, text=True,
)
if r.returncode != 0:
    raise RuntimeError(
        f"deploy-frontend.py failed (exit {r.returncode}).\n"
        f"stdout tail: {r.stdout[-2500:]}\nstderr tail: {r.stderr[-2500:]}"
    )
print(r.stdout[-600:])

## Step 10: Read Stack Outputs

Capture `AmplifyUrl`, `CognitoUserPoolId`, and `RuntimeArn` — the remaining notebooks need them.


In [ ]:
import os
import boto3

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or boto3.session.Session().region_name
)
if not REGION:
    raise RuntimeError(
        "No AWS region configured. Set AWS_REGION (or AWS_DEFAULT_REGION), "
        "or run `aws configure set region <region>`."
    )
cfn = boto3.client("cloudformation", region_name=REGION)

resp = cfn.describe_stacks(StackName="FAST-stack")
outputs = {o["OutputKey"]: o["OutputValue"] for o in resp["Stacks"][0]["Outputs"]}

AMPLIFY_URL = outputs.get("AmplifyUrl", "")
COGNITO_USER_POOL_ID = outputs.get("CognitoUserPoolId", "")
RUNTIME_ARN = outputs.get("RuntimeArn", "")

print(f"Amplify URL:            {AMPLIFY_URL}")
print(f"Cognito User Pool ID:   {COGNITO_USER_POOL_ID}")
print(f"Runtime ARN:            {RUNTIME_ARN}")

missing_outputs = [k for k, v in {
    "AmplifyUrl": AMPLIFY_URL,
    "CognitoUserPoolId": COGNITO_USER_POOL_ID,
    "RuntimeArn": RUNTIME_ARN,
}.items() if not v]
if missing_outputs:
    raise RuntimeError(
        f"FAST-stack outputs missing: {missing_outputs}. "
        f"Re-run the deploy cell or inspect CloudFormation events."
    )

## Step 11: Create a Cognito User

Create `workshop@example.com` with a temporary password. On first login the user will be
prompted to set a permanent password.


In [ ]:
import secrets, string
import boto3
from botocore.exceptions import ClientError

cognito = boto3.client("cognito-idp", region_name=REGION)

USERNAME = "workshop@example.com"
# Generate a random temporary password that satisfies Cognito's default
# policy (>=8 chars, upper + lower + number + symbol). The participant
# will be prompted to rotate it on first login anyway — this just avoids
# shipping a well-known hardcoded value in the workshop repo.
alphabet = string.ascii_letters + string.digits
TEMP_PASSWORD = "".join(secrets.choice(alphabet) for _ in range(20)) + "!Aa1"

try:
    cognito.admin_create_user(
        UserPoolId=COGNITO_USER_POOL_ID,
        Username=USERNAME,
        TemporaryPassword=TEMP_PASSWORD,
        UserAttributes=[
            {"Name": "email", "Value": USERNAME},
            {"Name": "email_verified", "Value": "true"},
        ],
    )
    print(f"Created user {USERNAME}")
    print("Temporary password set \u2014 retrieve from Secrets Manager / reset on first login (value not printed)")
except ClientError as e:
    if e.response["Error"]["Code"] == "UsernameExistsException":
        # User already exists; reset the temp password so the caller has a
        # known value for login.
        cognito.admin_set_user_password(
            UserPoolId=COGNITO_USER_POOL_ID,
            Username=USERNAME,
            Password=TEMP_PASSWORD,
            Permanent=False,
        )
        print(f"User {USERNAME} already existed — temporary password was reset")
        print("Temporary password set \u2014 retrieve from Secrets Manager / reset on first login (value not printed)")
    else:
        raise


## Test the Baseline

Open the **Amplify URL** printed above, log in with `workshop@example.com` and the temporary password printed above, and
send a message such as:

> Search for flights from SFO to Tokyo for 2026-09-15

The agent will respond but the `search_flights` tool call will fail — the gateway is not
wired up yet. That is expected. You will fix this in notebook 04a (MCP path) or 04b (AgentCore
path).

Proceed to **notebook 03 — Connect to LLM Gateway**.
